##**Installing Dependencies**

In [1]:
!pip install pytorch-msssim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

##**Importing Libraries**

In [2]:
import os
import glob
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pytorch_msssim import ssim

##**Dataset**

**Dataset Preparation**

In [3]:
class SuperResolutionDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        """
        Args:
            lr_dir (str): Path to the low-resolution .npy files folder.
            hr_dir (str): Path to the high-resolution .npy files folder.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        # Get sorted lists of .npy file paths from each folder.
        self.lr_images = sorted(glob.glob(os.path.join(lr_dir, '*.npy')))
        self.hr_images = sorted(glob.glob(os.path.join(hr_dir, '*.npy')))
        self.transform = transform

        if len(self.lr_images) != len(self.hr_images):
            raise ValueError("The number of LR and HR files do not match!")

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):
        lr_path = self.lr_images[idx]
        hr_path = self.hr_images[idx]

        # Load the .npy files.
        lr_array = np.load(lr_path)
        hr_array = np.load(hr_path)

        # Convert the numpy arrays to PyTorch tensors.
        lr_tensor = torch.tensor(lr_array, dtype=torch.float32)
        hr_tensor = torch.tensor(hr_array, dtype=torch.float32)

        # If the image is 2D (grayscale), add a channel dimension.
        if lr_tensor.ndim == 2:
            lr_tensor = lr_tensor.unsqueeze(0)
        if hr_tensor.ndim == 2:
            hr_tensor = hr_tensor.unsqueeze(0)

        if self.transform:
            lr_tensor = self.transform(lr_tensor)
            hr_tensor = self.transform(hr_tensor)

        return lr_tensor, hr_tensor

**Data Augmentation**

In [4]:
transform = transforms.Compose([
    transforms.ToPILImage(),                  # Convert tensor (from .npy) to PIL image
    transforms.RandomHorizontalFlip(p=0.5),   # 50% chance to flip horizontally
    transforms.RandomRotation(degrees=15),    # Random rotation between -15 and 15 degrees
    transforms.ToTensor(),                    # Convert back to tensor
    transforms.Normalize((0.5,), (0.5,))        # Normalize: update the mean and std if needed
])

**Defining Dataset Paths**

In [5]:
lr_dir = '/content/drive/MyDrive/Dataset/LR'
hr_dir = '/content/drive/MyDrive/Dataset/HR'

In [6]:
dataset = SuperResolutionDataset(lr_dir, hr_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


##**Modelling**

**Defining SRCNN model**

In [7]:
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, padding=0)
        # Change the output channels of conv3 to 4
        self.conv3 = nn.Conv2d(32, 4, kernel_size=5, padding=2)
        # Add an upsampling layer (e.g., using PixelShuffle)
        self.upsample = nn.PixelShuffle(2)  # Upscales by a factor of 2

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        # Apply upsampling
        x = self.upsample(x)
        return x

**Preparation of SCRNN model**

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SRCNN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

**Training the Model**

In [9]:
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for lr_imgs, hr_imgs in dataloader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)

        optimizer.zero_grad()
        outputs = model(lr_imgs)
        loss = criterion(outputs, hr_imgs)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.6f}")

Epoch 1/50, Loss: 0.021255
Epoch 2/50, Loss: 0.013882
Epoch 3/50, Loss: 0.013392
Epoch 4/50, Loss: 0.013298
Epoch 5/50, Loss: 0.013203
Epoch 6/50, Loss: 0.013379
Epoch 7/50, Loss: 0.013042
Epoch 8/50, Loss: 0.013223
Epoch 9/50, Loss: 0.013012
Epoch 10/50, Loss: 0.013106
Epoch 11/50, Loss: 0.012844
Epoch 12/50, Loss: 0.013465
Epoch 13/50, Loss: 0.012907
Epoch 14/50, Loss: 0.013003
Epoch 15/50, Loss: 0.012830
Epoch 16/50, Loss: 0.012736
Epoch 17/50, Loss: 0.012674
Epoch 18/50, Loss: 0.012762
Epoch 19/50, Loss: 0.012768
Epoch 20/50, Loss: 0.012688
Epoch 21/50, Loss: 0.012583
Epoch 22/50, Loss: 0.013072
Epoch 23/50, Loss: 0.012759
Epoch 24/50, Loss: 0.012707
Epoch 25/50, Loss: 0.012621
Epoch 26/50, Loss: 0.012443
Epoch 27/50, Loss: 0.012573
Epoch 28/50, Loss: 0.012376
Epoch 29/50, Loss: 0.012373
Epoch 30/50, Loss: 0.012352
Epoch 31/50, Loss: 0.012465
Epoch 32/50, Loss: 0.012373
Epoch 33/50, Loss: 0.012371
Epoch 34/50, Loss: 0.012325
Epoch 35/50, Loss: 0.012336
Epoch 36/50, Loss: 0.012222
E

**Evaluating the model**

In [10]:
model.eval()
total_mse = 0.0
total_psnr = 0.0
total_ssim = 0.0
num_batches = 0

with torch.no_grad():
    for lr_imgs, hr_imgs in dataloader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)
        outputs = model(lr_imgs)

        # Compute MSE for this batch
        mse_val = criterion(outputs, hr_imgs).item()
        total_mse += mse_val

        # Compute PSNR (assuming pixel values in [0, 1])
        psnr_val = 10 * math.log10(1 / mse_val) if mse_val > 0 else float('inf')
        total_psnr += psnr_val

        # Compute SSIM using the pytorch_msssim package
        ssim_val = ssim(outputs, hr_imgs, data_range=1, size_average=True).item()
        total_ssim += ssim_val

        num_batches += 1

avg_mse = total_mse / num_batches
avg_psnr = total_psnr / num_batches
avg_ssim = total_ssim / num_batches

print("\nEvaluation Metrics:")
print(f"Average MSE: {avg_mse:.6f}")
print(f"Average PSNR: {avg_psnr:.2f} dB")
print(f"Average SSIM: {avg_ssim:.4f}")


Evaluation Metrics:
Average MSE: 0.011876
Average PSNR: 19.34 dB
Average SSIM: 0.8216


**Saving the model**

In [ ]:
# Define the path where the model weights will be saved
model_path = "/content/scrnn_weights3a"

# Save only the model's state_dict (recommended)
torch.save(model.state_dict(), model_path)

print(f"Model weights saved to {model_path}")